# S23DR 2026 — Procedural Roof Reconstruction

Geometry-first pipeline: no neural network weights needed.

**Steps**
1. Install dependencies & pull repo
2. Load validation dataset
3. Run procedural pipeline on one sample (debug)
4. Full evaluation: HSS over all 1024 validation samples
5. Visualise: point cloud + GT + procedural wireframe

In [ ]:
!pip install -q datasets huggingface_hub scipy numpy shapely

import os
REPO_DIR = "/content/3d_building_construction"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/12turtleships/3d_building_construction {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import io, zipfile
import numpy as np
from datasets import load_dataset
from s23dr.metrics import hss
from s23dr.procedural import reconstruct_to_segments

print("Imports OK")

In [ ]:
HF_DATASET = "usm3d/s23dr-2026-sampled_4096_v2"
SPLIT      = "validation"

def _unpack(row):
    out = {}
    with zipfile.ZipFile(io.BytesIO(row["data"])) as zf:
        for name in zf.namelist():
            if name.endswith(".npy"):
                out[name[:-4]] = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
    out["order_id"] = row.get("order_id", "")
    return out

rows = [_unpack(r) for r in load_dataset(HF_DATASET, split=SPLIT)]
print(f"Loaded {len(rows)} validation samples")
print(f"Keys in sample 0: {list(rows[0].keys())}")

In [ ]:
# ── Debug: run pipeline on one sample ────────────────────────────────────────
import time

SAMPLE_IDX = 0
r = rows[SAMPLE_IDX]

xyz      = r["xyz_norm"]       # (N, 3)
cid      = r["class_id"]       # (N,)
gt_segs  = r["gt_segments"]    # (E, 2, 3)

t0 = time.time()
pred_segs = reconstruct_to_segments(xyz, class_id=cid)
elapsed = time.time() - t0

scores = hss(pred_segs, gt_segs)

print(f"order_id     : {r['order_id']}")
print(f"xyz shape    : {xyz.shape}")
print(f"GT segments  : {len(gt_segs)}")
print(f"Pred segments: {len(pred_segs)}")
print(f"Time         : {elapsed*1000:.0f} ms")
print(f"HSS          : {scores['hss']:.4f}")
print(f"Precision    : {scores['precision']:.4f}")
print(f"Recall       : {scores['recall']:.4f}")

In [ ]:
# ── Full evaluation over all 1024 validation samples ─────────────────────────
import time

results = []
t0 = time.time()

for i, r in enumerate(rows):
    xyz     = r["xyz_norm"]
    cid     = r["class_id"]
    gt_segs = r["gt_segments"]

    pred_segs = reconstruct_to_segments(xyz, class_id=cid)
    scores    = hss(pred_segs, gt_segs)
    results.append({"order_id": r["order_id"], **scores})

    if (i + 1) % 100 == 0:
        mean_h = np.mean([r["hss"] for r in results])
        print(f"  [{i+1:4d}/1024]  mean_hss={mean_h:.4f}  ({time.time()-t0:.0f}s)")

hss_arr  = np.array([r["hss"]       for r in results])
prec_arr = np.array([r["precision"] for r in results])
rec_arr  = np.array([r["recall"]    for r in results])

print("=" * 45)
print(f"  Samples   : {len(results)}")
print(f"  HSS       : {hss_arr.mean():.4f}  (std {hss_arr.std():.4f})")
print(f"  Precision : {prec_arr.mean():.4f}")
print(f"  Recall    : {rec_arr.mean():.4f}")
print("=" * 45)

In [ ]:
# ── Visualise: point cloud + GT wireframe + procedural wireframe ──────────────
import plotly.graph_objects as go

SAMPLE_IDX = 0   # ← change to inspect different samples
MAX_PTS    = 2048

r        = rows[SAMPLE_IDX]
xyz_np   = r["xyz_norm"]
gt_segs  = r["gt_segments"]
pred_segs = reconstruct_to_segments(xyz_np, class_id=r["class_id"])
scores   = hss(pred_segs, gt_segs)

rng = np.random.default_rng(0)
vis = rng.choice(len(xyz_np), min(MAX_PTS, len(xyz_np)), replace=False)
pc  = xyz_np[vis]

pc_trace = go.Scatter3d(
    x=pc[:,0], y=pc[:,1], z=pc[:,2],
    mode="markers",
    marker=dict(size=1.5, color="royalblue", opacity=0.35),
    name="Point cloud",
)

def _seg_trace(segs, color, name, width=4):
    if len(segs) == 0:
        return go.Scatter3d(x=[], y=[], z=[], mode="lines",
                            line=dict(color=color, width=width), name=name)
    xs, ys, zs = [], [], []
    for s in segs:
        xs += [float(s[0,0]), float(s[1,0]), None]
        ys += [float(s[0,1]), float(s[1,1]), None]
        zs += [float(s[0,2]), float(s[1,2]), None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                        line=dict(color=color, width=width), name=name)

fig = go.Figure(data=[
    pc_trace,
    _seg_trace(gt_segs,   "limegreen", f"GT ({len(gt_segs)} segs)"),
    _seg_trace(pred_segs, "red",       f"Procedural ({len(pred_segs)} segs)"),
])
fig.update_layout(
    title=(f"Sample {SAMPLE_IDX} | {r['order_id']} | "
           f"HSS={scores['hss']:.3f}  P={scores['precision']:.3f}  R={scores['recall']:.3f}"),
    scene=dict(aspectmode="data"),
    height=680,
    margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()
print(f"GT: {len(gt_segs)} segs  |  Pred: {len(pred_segs)} segs")